# CAB: live timing on a GPU, part 1 (PROTOCOL.md, protocol v3, addendum C)

Times the standard estimators (full budget) and the two certified CAB configurations on a GPU, on the same 300 test pairs per set as the CPU live timing. Nothing is trained here: the bundle holds the certified controllers and gates as plain arrays.

**Steps**
1. Upload `gpu_bundle_part1.zip` to the top level of your Google Drive ("My Drive").
2. Choose a GPU runtime: in Colab, *Runtime > Change runtime type > T4 GPU*; in VS Code with the Colab extension, select a Colab **GPU** server as the kernel.
3. *Run all*. The second cell asks for the GitHub token right away and checks it (a fine-grained token restricted to this repository with *Contents: Read and write*; or store it as the Colab secret `GITHUB_TOKEN`). Then everything runs unattended: phase 1 (SC2-PCR and TurboReg, 4 test sets: fresh blind and held-out scenes) takes about 25 minutes on a T4, phase 2 (MAC on the fresh blind test; its clique search runs on the CPU, as in the authors' code) about 1-1.5 hours.
4. After each phase the results are copied to Drive and published to the GitHub repository (`results/gpu/`). Never put the token in this notebook's code.

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: select a GPU runtime first (see step 2)"
print(torch.cuda.get_device_name(0), "| torch", torch.__version__, "| CUDA", torch.version.cuda)

In [ ]:
# GitHub token, asked once at the start and checked now, used at the end of each phase to publish the results
import getpass, json, urllib.request
REPO = "AFAcoding/Certified-Adaptive-Budgets-for-Robust-Point-Cloud-Registration"
try:
    from google.colab import userdata
    _token = userdata.get("GITHUB_TOKEN")
except Exception:
    _token = None
if not _token:
    _token = getpass.getpass("GitHub token (fine-grained, this repository, Contents: read and write): ").strip()
_req = urllib.request.Request(f"https://api.github.com/repos/{REPO}",
                              headers={"Authorization": f"Bearer {_token}", "Accept": "application/vnd.github+json"})
with urllib.request.urlopen(_req) as _r:
    print("token OK for", json.load(_r)["full_name"], "- you can leave the notebook running")

In [ ]:
%pip install -q igraph

In [ ]:
import os, zipfile
ZIP = "/content/drive/MyDrive/gpu_bundle_part1.zip"
if not os.path.exists("/content/gpu_bundle_part1/config.json"):
    if not os.path.exists(ZIP):
        # VS Code: if this fails, run the command "Colab: Mount Google Drive to Server", then run this cell again
        from google.colab import drive
        drive.mount("/content/drive")
    with zipfile.ZipFile(ZIP) as z:
        z.extractall("/content")
print(sorted(os.listdir("/content/gpu_bundle_part1")))

In [ ]:
import base64, gzip, shutil, time

def publish(path):
    """Copy the results to Drive and publish them (gzipped) to the GitHub repository, results/gpu/."""
    if os.path.isdir("/content/drive/MyDrive"):
        shutil.copy(path, "/content/drive/MyDrive/")
        print("copied to Drive:", os.path.basename(path))
    name = f"results/gpu/{os.path.basename(path)[:-5]}_{time.strftime('%Y%m%d_%H%M%S')}.json.gz"
    body = json.dumps({"message": f"GPU timing results: {os.path.basename(name)}",
                       "content": base64.b64encode(gzip.compress(open(path, "rb").read())).decode()}).encode()
    req = urllib.request.Request(f"https://api.github.com/repos/{REPO}/contents/{name}", data=body, method="PUT",
                                 headers={"Authorization": f"Bearer {_token}", "Accept": "application/vnd.github+json",
                                          "X-GitHub-Api-Version": "2022-11-28"})
    with urllib.request.urlopen(req) as r:
        print("published:", json.load(r)["content"]["html_url"])

## Phase 1: SC2-PCR and TurboReg on every timed test set

In [ ]:
!cd /content/gpu_bundle_part1 && python -u run_gpu_timing.py --bundle . --estimators sc2,turbo --out /content/cab_gpu_results_sc2_turbo_part1.json
publish("/content/cab_gpu_results_sc2_turbo_part1.json")

## Phase 2: MAC on the fresh blind test (graphs on the GPU, clique search on the CPU)

In [ ]:
!cd /content/gpu_bundle_part1 && python -u run_gpu_timing.py --bundle . --estimators mac --sets th,tl --out /content/cab_gpu_results_mac_part1.json
publish("/content/cab_gpu_results_mac_part1.json")

## Part 2: picked up automatically
Waits (up to 6 hours) for `gpu_bundle_part2.zip` in your Drive, then times SC2-PCR and TurboReg on its six test sets and publishes the results. Just upload the zip to "My Drive" when it is ready.

In [ ]:
import subprocess, sys
ZIP2 = "/content/drive/MyDrive/gpu_bundle_part2.zip"
t0 = time.time()
while time.time() - t0 < 6 * 3600:
    try:
        with zipfile.ZipFile(ZIP2) as z:  # fails until the upload is complete
            z.extractall("/content")
        break
    except Exception:
        time.sleep(60)
else:
    raise SystemExit("gpu_bundle_part2.zip did not appear in Drive within 6 hours")
print("part 2 found after", round((time.time() - t0) / 60), "min")
proc = subprocess.Popen([sys.executable, "-u", "run_gpu_timing.py", "--bundle", ".", "--estimators", "sc2,turbo",
                         "--out", "/content/cab_gpu_results_sc2_turbo_part2.json"], cwd="/content/gpu_bundle_part2",
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:  # progress shown in this cell
    print(line, end="")
assert proc.wait() == 0, "part 2 failed"
publish("/content/cab_gpu_results_sc2_turbo_part2.json")